## MNIST training

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fpschill/cas_main_dl_2026/blob/main/03_mnist_training.ipynb)

We use **Keras 3** with the **PyTorch** backend. The backend is selected with the environment variable `KERAS_BACKEND`, which has to be set *before* `keras` is imported (see the first lines of the next cell). If you change it, restart the kernel.

In [ ]:
# Imports
import os
os.environ["KERAS_BACKEND"] = "torch"  # must be set BEFORE keras is imported
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import keras

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())
print("GPU available:", torch.cuda.is_available())

In [ ]:
def plotTraining(history):
    # summarize history for accuracy
    plt.figure(figsize=(14,6))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'valid'], loc='lower right')
    plt.subplot(1,2,2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'valid'], loc='upper right')
    plt.show()

In [ ]:
# hello world example

mnist = keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

model = keras.models.Sequential([
  keras.layers.Input(shape=(28,28)),
  keras.layers.Flatten(),
  keras.layers.Dense(64, activation='relu'),
  keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=5)
plotTraining(history)

model.evaluate(x_test, y_test)

## Activation Functions

- Different Types of Activation Functions. What do you observe? Training speed, final performance, etc...
- Why do we need them at all?

In [ ]:
# no activation function (linear)
# https://keras.io/api/layers/activations/

mnist = keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

model = keras.models.Sequential([
  keras.layers.Input(shape=(28,28)),
  keras.layers.Flatten(),

  #PLAY AROUND HERE

  keras.layers.Dense(64, activation='relu'),
  #keras.layers.Dense(64), #linear
  #keras.layers.Dense(64, activation='sigmoid'),
  keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3)
plotTraining(history)

model.evaluate(x_test, y_test)

## Optimization - Weight initialization

In [ ]:
# weight initialization
# https://keras.io/api/layers/initializers/

mnist = keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

model = keras.models.Sequential([
  keras.layers.Input(shape=(28,28)),
  keras.layers.Flatten(),

  #PLAY AROUND HERE

  keras.layers.Dense(64, activation='relu'),
  #keras.layers.Dense(64, activation='relu', kernel_initializer=keras.initializers.Zeros()),
  #keras.layers.Dense(64, activation='relu', kernel_initializer=keras.initializers.Constant(value=-1)),
  #keras.layers.Dense(64, activation='relu', kernel_initializer=keras.initializers.Constant(value=1)),

  keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3)
plotTraining(history)

model.evaluate(x_test, y_test)

## Optimization - Stochastic Gradient Descent

In [ ]:
# optimizer & stochastic gradient descent
# https://keras.io/api/optimizers/

mnist = keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

model = keras.models.Sequential([
  keras.layers.Input(shape=(28,28)),
  keras.layers.Flatten(),
  keras.layers.Dense(64, activation='relu'),
  keras.layers.Dense(10, activation='softmax')
])

#PLAY AROUND HERE
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
#model.compile(optimizer='sgd', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=10)
#history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3, batch_size=1) #sgd
#history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3, batch_size=len(x_train)) #batch gradient descent
#history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3, batch_size=64) #32 default
plotTraining(history)

model.evaluate(x_test, y_test)

## Data Augmentation to increase performance

In [ ]:
#no data augmentation
mnist = keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Make train data smaller -- only 100 examples!
np.random.seed(36)
train_data_idx=np.random.choice(range(0,len(x_train)),100,replace=False)
x_train=x_train[train_data_idx]
y_train=y_train[train_data_idx]

x_train=x_train.reshape((len(x_train),28,28,1))
x_test=x_test.reshape((len(x_test),28,28,1))

model = keras.models.Sequential([
  keras.layers.Input(shape=(28,28,1)),
  keras.layers.Flatten(),
  keras.layers.Dense(64, activation='relu'),
  keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(x_train, y_train, epochs=50, batch_size=32, validation_data=(x_test, y_test))
plotTraining(history)

model.evaluate(x_test, y_test)

In Keras 3, data augmentation is done with **preprocessing layers** such as `RandomRotation` and `RandomTranslation` (the old `ImageDataGenerator` no longer exists). We put these layers at the start of the model. They change every training batch randomly, but they are **only active during training** (`fit`); in `evaluate` and `predict` they do nothing, so the test images are not changed.

To look at augmented images, we call the layers directly with `training=True`.

In [ ]:
# do data aug
# https://keras.io/api/layers/preprocessing_layers/image_augmentation/
data_augmentation = keras.Sequential([
    keras.layers.RandomRotation(15/360, fill_mode='nearest'),                # rotate by up to +-15 degrees (factor is a fraction of 360)
    keras.layers.RandomTranslation(0.15, 0.15, fill_mode='nearest'),        # shift by up to +-15% of the height/width
], name='data_augmentation')

i=2
plt.imshow(x_train[i,:,:,0],cmap="gray")
# original image

In [ ]:
# augmented image
plt.figure(figsize=(15,15))
for j in range (0,25):
  plt.subplot(5,5,j+1)
  x_aug = data_augmentation(x_train[i:(i+1)], training=True)
  plt.imshow(keras.ops.convert_to_numpy(x_aug)[0,:,:,0],cmap="gray")

In [ ]:
mnist = keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Make train data smaller -- only 100 examples!
np.random.seed(36)
train_data_idx=np.random.choice(range(0,len(x_train)),100,replace=False)
x_train=x_train[train_data_idx]
y_train=y_train[train_data_idx]

x_train=x_train.reshape((len(x_train),28,28,1))
x_test=x_test.reshape((len(x_test),28,28,1))

model = keras.models.Sequential([
  keras.layers.Input(shape=(28,28,1)),
  data_augmentation,  # only active during training
  keras.layers.Flatten(),
  keras.layers.Dense(64, activation='relu'),
  keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# fits the model on batches with real-time data augmentation:
history = model.fit(x_train, y_train, batch_size=32, epochs=300, validation_data=(x_test, y_test))
plotTraining(history)

model.evaluate(x_test, y_test)

### Accuracy on the test set is around 10% better if you use data augmentation
0.7491 -> 0.8349
### Also: much less overfitting!

## Other Architecture: CNN

model from scratch 

In [ ]:
from keras.models import Sequential, Model
from keras.layers import Dense, Flatten, Input
from keras.layers import Conv2D, MaxPooling2D
from keras.datasets import mnist

(x_digits_train, y_digits_train), (x_digits_test, y_digits_test) = mnist.load_data()

# Make train data smaller
np.random.seed(36)
train_data_idx=np.random.choice(range(0,len(x_digits_train)),100,replace=False)
x_digits_train=x_digits_train[train_data_idx]
y_digits_train=y_digits_train[train_data_idx]

# Preprocess data
x_digits_train = x_digits_train.astype('float32')
x_digits_test = x_digits_test.astype('float32')
x_digits_train = x_digits_train/ 255
x_digits_test = x_digits_test/ 255
x_digits_train=x_digits_train.reshape((len(x_digits_train),28,28,1))
x_digits_test=x_digits_test.reshape((len(x_digits_test),28,28,1))


# Define model
model_digits = Sequential()
model_digits.add(Input(shape=(28,28,1)))
model_digits.add(Conv2D(8,(3,3),activation='relu'))
model_digits.add(Conv2D(8,(3,3),activation='relu'))
model_digits.add(MaxPooling2D((2,2)))
model_digits.add(Conv2D(16,(3,3),activation='relu'))
model_digits.add(Conv2D(16,(3,3),activation='relu'))
model_digits.add(Flatten())
model_digits.add(Dense(50, activation='relu'))
model_digits.add(Dense(10, activation='softmax'))
# Compile model
model_digits.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# train model
history=model_digits.fit(x_digits_train, y_digits_train,validation_data=(x_digits_test, y_digits_test),
                         batch_size=128, epochs=80)
plotTraining(history)

model_digits.evaluate(x_digits_test,y_digits_test)

### Strong overfitting!

## CNN + Data-Augmentation

In [ ]:
# put the augmentation layers in front of the CNN from above and continue training it
model_digits_aug = Sequential([Input(shape=(28,28,1)), data_augmentation, model_digits])
model_digits_aug.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# fits the model on batches with real-time data augmentation:
history=model_digits_aug.fit(x_digits_train, y_digits_train, batch_size=64, validation_data=(x_digits_test, y_digits_test), epochs=120)
plotTraining(history)

model_digits_aug.evaluate(x_digits_test,y_digits_test)

### Test performance summary (so far)

- simple model:
         0.74 -> 0.83
  
- CNN:
        0.76 -> 0.88

## CNN + Fine-Tuning

- train a model on letters (E-MNIST)
- fix lower layers and fine-tune to digits (MNIST)

In [ ]:
# Downloading the data, if it does not exist (takes some time)
import os,sys,zipfile
if "google.colab" in sys.modules:
    %pip install wget
import wget

if not(os.path.isfile("matlab.zip")):
    filename=wget.download("https://biometrics.nist.gov/cs_links/EMNIST/matlab.zip", "matlab.zip")
if not(os.path.isdir("./matlab")):
    zf = zipfile.ZipFile(os.path.join(".","matlab.zip"), "r")
    zf.extractall()

In [ ]:
#load emnist data
from scipy import io as spio

emnist = spio.loadmat("matlab/emnist-letters.mat")

# load training dataset
x_letter_train = emnist["dataset"][0][0][0][0][0][0]
y_letter_train = emnist["dataset"][0][0][0][0][0][1]

# load test dataset
x_letter_test = emnist["dataset"][0][0][1][0][0][0]
y_letter_test = emnist["dataset"][0][0][1][0][0][1]

x_letter_train = x_letter_train.reshape(x_letter_train.shape[0], 28, 28, 1, order="A")
x_letter_test = x_letter_test.reshape(x_letter_test.shape[0], 28, 28, 1, order="A")
x_letter_train = x_letter_train.astype('float32')
x_letter_test = x_letter_test.astype('float32')
x_letter_train /= 255
x_letter_test /= 255

In [ ]:
# Define model
model_letters = Sequential()
model_letters.add(Input(shape=(28,28,1)))
model_letters.add(Conv2D(8,(3,3), activation='relu'))
model_letters.add(Conv2D(8,(3,3), activation='relu'))
model_letters.add(MaxPooling2D((2,2)))
model_letters.add(Conv2D(16,(3,3), activation='relu'))
model_letters.add(Conv2D(16,(3,3), activation='relu'))
model_letters.add(MaxPooling2D((2,2)))
model_letters.add(Flatten(name="Flat"))
model_letters.add(Dense(50, activation='relu'))
model_letters.add(Dense(27, activation='softmax'))
# Compile model
model_letters.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# train model
history=model_letters.fit(x_letter_train, y_letter_train,validation_data=(x_letter_test, y_letter_test),
                         batch_size=128, epochs=5, verbose=1)
plotTraining(history)

model_letters.evaluate(x_letter_test,y_letter_test)
model_letters.summary()

In [ ]:
letter_model = Model(inputs=model_letters.inputs, outputs=model_letters.get_layer("Flat").output)
letter_model.summary()

In [ ]:
x = letter_model.output
# add a hidden and the new output layer
x = Dense(50, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)

model = Model(inputs=letter_model.input, outputs=predictions)
model.summary()

In [ ]:
#fix the lower layers
for layer in letter_model.layers:
    layer.trainable = False
for i, layer in enumerate(model.layers):
    print(i, layer.name,layer.trainable)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
#train
history=model.fit(x_digits_train, y_digits_train,validation_data=(x_digits_test, y_digits_test),
                         batch_size=64, epochs=80)
plotTraining(history)

model.evaluate(x_digits_test,y_digits_test)

### Test performance summary (so far)

- simple model:
 
         0.7491 -> 0.8349 (data aug.)
    
    
- CNN (for other task, much much higher!):
    
        0.7685 -> 0.8414 (data aug.)
        0.7685 -> 0.8974 (fine-tuning)

##  CNN + Data augmentation and Fine-Tuning

In [ ]:
letter_model = Model(inputs=model_letters.inputs, outputs=model_letters.get_layer("Flat").output)

x = letter_model.output
x = Dense(50, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)
model = Model(inputs=letter_model.input, outputs=predictions)
for layer in letter_model.layers:
    layer.trainable = False

# put the augmentation layers in front of the model
model_aug = Sequential([Input(shape=(28,28,1)), data_augmentation, model])
model_aug.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])

# fits the model on batches with real-time data augmentation:
history=model_aug.fit(x_digits_train, y_digits_train, batch_size=64, validation_data=(x_digits_test, y_digits_test), epochs=120)
plotTraining(history)

model_aug.evaluate(x_digits_test,y_digits_test)

### Test performance summary
**trained using 100 images only!**

- simple model:
 
         0.7491 -> 0.8349 (data aug.)
    
- CNN (for other task, much much higher!):
    
        0.7685 -> 0.8414 (data aug.)
        0.7685 -> 0.8974 (fine-tuning)
        0.7685 -> 0.9251 (data aug. & fine-tuning)

## Error analysis

In [ ]:
## same network from scratch using all training data

(x_digits_train, y_digits_train), (x_digits_test, y_digits_test) = mnist.load_data()


# Preprocess data
x_digits_train = x_digits_train.astype('float32')
x_digits_test = x_digits_test.astype('float32')
x_digits_train = x_digits_train/ 255
x_digits_test = x_digits_test/ 255
x_digits_train=x_digits_train.reshape((len(x_digits_train),28,28,1))
x_digits_test=x_digits_test.reshape((len(x_digits_test),28,28,1))


# Define model
model_digits_full = Sequential()
model_digits_full.add(Input(shape=(28,28,1)))
model_digits_full.add(Conv2D(16,(3,3),activation='relu'))
model_digits_full.add(Conv2D(16,(3,3),activation='relu'))
model_digits_full.add(MaxPooling2D((2,2)))
model_digits_full.add(Conv2D(32,(3,3),activation='relu'))
model_digits_full.add(Conv2D(32,(3,3),activation='relu'))

model_digits_full.add(Flatten())
model_digits_full.add(Dense(50, activation='relu'))
model_digits_full.add(Dense(10, activation='softmax'))
# Compile model
model_digits_full.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# train model
history=model_digits_full.fit(x_digits_train, y_digits_train,validation_data=(x_digits_test, y_digits_test),
                         batch_size=128, epochs=5, verbose=1)
plotTraining(history)

model_digits_full.evaluate(x_digits_test,y_digits_test)

In [ ]:
pred_full=np.argmax(model_digits_full.predict(x_digits_test), axis=1)
wrong_idx=np.where(pred_full!=y_digits_test)[0]
len(wrong_idx)

In [ ]:
plt.figure(figsize=(20,40))
for i in range(0,50):
  plt.subplot(10,5,i+1)
  plt.imshow(x_digits_test[wrong_idx[i],:,:,0],cmap="gray")
  plt.title("pred: "+str(pred_full[wrong_idx[i]])+ " true: "+str(y_digits_test[wrong_idx[i]]))